# 이상지질혈증 유병 예측 — CatBoost 시드 앙상블

- 방식: 동일 파라미터(best_params.json) + 다중 시드 → OOF proba 평균 앙상블
- Train/Test 8:2 Hold-out → Train 내부 5-Fold OOF
- Threshold: RECALL_MIN 0.85, F1 최대화
- 원본 파일: `CatBoost_Optuna_DL_FE.py`

In [ ]:
"""
고혈압유병 예측 — CatBoost 시드 앙상블
Python 3.13 | catboost>=1.2 | scikit-learn>=1.4

방식: 동일 파라미터(best_params.json) + 다중 시드 → OOF proba 평균 앙상블
      Train/Test 8:2 Hold-out → Train 내부 5-Fold OOF
      Threshold: RECALL_MIN 0.85, F1 최대화
"""

import json
import os
import warnings
from typing import Any

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from numpy.typing import NDArray
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")

# ── 경로 설정 ─────────────────────────────────────────────────
DATA_PATH: str = "/Users/admin/PycharmProjects/AH_03_03/ai_worker/data/hn_all_preprocessed.csv"
BASE_MODEL_DIR: str = "/Users/admin/PycharmProjects/AH_03_03/ai_worker/ml/CAT15~24/outputs/optuna_DL_FE"
OUTPUT_DIR: str = "/Users/admin/PycharmProjects/AH_03_03/ai_worker/ml/CAT15~24/outputs/seed_ensemble_DL"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 설정 ──────────────────────────────────────────────────────
TARGET: str = "이상지질혈증유병"
N_SPLITS: int = 5
BASE_SEED: int = 42
N_TRIALS: int = 50
RECALL_MIN: float = 0.85
THRESHOLD_RANGE: NDArray[np.float64] = np.arange(0.30, 0.71, 0.01)

# ── 앙상블 시드 목록 ──────────────────────────────────────────
ENSEMBLE_SEEDS: list[int] = [42, 123, 2024, 777, 2025]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FE (원본 파일과 동일하게 유지)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Feature Engineering 설정 및 함수

In [ ]:
USE_AGE_BIN: bool = True
USE_BMI_BIN: bool = True
USE_WEIGHT_BIN: bool = False
USE_ALCOHOL_RISK: bool = False
USE_WALK_LEVEL: bool = False
USE_STRENGTH: bool = False
USE_FAMILY_SUM: bool = True
USE_BMI_X_AGE: bool = True
USE_OBESITY_FLAG: bool = False


def apply_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    if USE_AGE_BIN:
        age_bins = [0, 40, 50, 60, 70, 80, 999]
        age_labels = ["나이_19_39", "나이_40대", "나이_50대", "나이_60대", "나이_70대", "나이_80이상"]
        df["_나이구간"] = pd.cut(df["나이"], bins=age_bins, labels=age_labels, right=False)
        for label in age_labels:
            df[label] = (df["_나이구간"] == label).astype(int)
        df = df.drop(columns=["_나이구간"])

    if USE_BMI_BIN:
        df["BMI_구간"] = pd.cut(df["BMI"], bins=[0, 23, 25, 30, 999], labels=[0, 1, 2, 3], right=False).astype(float)

    if USE_WEIGHT_BIN:
        wt_bins = [0, 50, 70, 90, 999]
        wt_labels = ["체중_저체중", "체중_정상", "체중_과체중", "체중_비만"]
        df["_체중구간"] = pd.cut(df["체중"], bins=wt_bins, labels=wt_labels, right=False)
        for label in wt_labels:
            df[label] = (df["_체중구간"] == label).astype(float)
        df = df.drop(columns=["_체중구간"])

    if USE_ALCOHOL_RISK:
        df["음주위험군"] = pd.cut(df["음주빈도"], bins=[-1, 0, 2, 99], labels=[0, 1, 2], right=True).astype(float)

    if USE_WALK_LEVEL:
        df["걷기활동량"] = pd.cut(df["걷기일수"], bins=[-1, 0, 3, 99], labels=[0, 1, 2], right=True).astype(float)

    if USE_STRENGTH:
        df["근력활동량"] = pd.cut(df["근력운동일수"], bins=[-1, 0, 2, 99], labels=[0, 1, 2], right=True).astype(float)

    if USE_FAMILY_SUM:
        df["고혈압가족력_합산"] = (
            df["고혈압가족력_부"].fillna(0) + df["고혈압가족력_모"].fillna(0) + df["고혈압가족력_형제"].fillna(0)
        ).clip(0, 3)
        df["당뇨가족력_합산"] = (
            df["당뇨가족력_부"].fillna(0) + df["당뇨가족력_모"].fillna(0) + df["당뇨가족력_형제"].fillna(0)
        ).clip(0, 3)
        df["고지혈증가족력_합산"] = (
            df["고지혈증가족력_부"].fillna(0) + df["고지혈증가족력_모"].fillna(0) + df["고지혈증가족력_형제"].fillna(0)
        ).clip(0, 3)

    if USE_BMI_X_AGE:
        df["BMI_X_나이"] = df["BMI"] * df["나이"]

    if USE_OBESITY_FLAG:
        df["비만여부"] = (df["BMI"] >= 25).astype(float)
        df.loc[df["BMI"].isna(), "비만여부"] = np.nan

    return df


# ── Threshold 탐색 ────────────────────────────────────────────

## 2. Threshold 탐색 함수

In [ ]:
def tune_threshold(
    proba: NDArray[np.float64],
    y_true: NDArray[np.int64],
) -> tuple[float, float, float, float]:
    best_f1: float = 0.0
    best_thr: float = 0.30
    for thr in THRESHOLD_RANGE:
        pred = (proba >= thr).astype(int)
        r = recall_score(y_true, pred)
        f = f1_score(y_true, pred, zero_division=0)
        if r >= RECALL_MIN and f > best_f1:
            best_f1 = f
            best_thr = round(float(thr), 2)
    final_pred = (proba >= best_thr).astype(int)
    final_recall = recall_score(y_true, final_pred)
    final_prec = precision_score(y_true, final_pred, zero_division=0)
    return best_thr, best_f1, final_recall, final_prec


# ── 단일 시드 OOF 학습 ────────────────────────────────────────

## 3. 단일 시드 OOF 학습 함수

In [ ]:
def train_single_seed(
    best_params: dict[str, Any],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    ratio: float,
    seed: int,
) -> tuple[NDArray[np.float64], list[CatBoostClassifier]]:
    """단일 시드로 5-Fold OOF 학습 → oof_proba 반환"""
    params: dict[str, Any] = {
        **best_params,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "class_weights": {0: 1.0, 1: ratio},
        "early_stopping_rounds": 50,
        "random_seed": seed,  # ← 시드만 변경
        "verbose": False,
        "allow_writing_files": False,
    }

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_proba: NDArray[np.float64] = np.zeros(len(y_train))
    fold_models: list[CatBoostClassifier] = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(Pool(X_tr, y_tr), eval_set=Pool(X_val, y_val))

        proba = model.predict_proba(X_val)[:, 1]
        oof_proba[val_idx] = proba
        fold_models.append(model)

        pred = (proba >= 0.5).astype(int)
        print(
            f"    Seed {seed} | Fold {fold} | "
            f"AUC: {roc_auc_score(y_val, proba):.4f} | "
            f"Recall: {recall_score(y_val, pred):.4f} | "
            f"F1: {f1_score(y_val, pred):.4f} | "
            f"iter: {model.best_iteration_}"
        )

    return oof_proba, fold_models

## 4. 데이터 로드 & Feature Engineering

In [ ]:
# ── 데이터 로드 ───────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"[0] 데이터 로드 | shape: {df.shape}")
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print(f"[0] {TARGET} 결측 제거 후 | shape: {df.shape}")

df = apply_feature_engineering(df)

drop_cols = [c for c in ["고혈압유병", "당뇨유병", "이상지질혈증유병", "비만단계"] if c in df.columns]
X: pd.DataFrame = df.drop(columns=drop_cols)
y: pd.Series = df[TARGET].astype(int)
print(f"\n[1] Feature 수: {X.shape[1]}")

## 5. Train/Test 분리 & 클래스 비율 확인

In [ ]:
# ── Train/Test 분리 (BASE_SEED 고정) ─────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=BASE_SEED, stratify=y)
print(f"\n[2] Train: {X_train.shape} | Test: {X_test.shape}")
print(f"    Train 양성 비율: {y_train.mean():.4f} | Test 양성 비율: {y_test.mean():.4f}")

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
ratio = float(neg / pos)
print(f"    class_weights: {{0: 1.0, 1: {ratio:.4f}}}")

## 6. best_params 로드

In [ ]:
# ── best_params 로드 ──────────────────────────────────────
best_params_path = os.path.join(BASE_MODEL_DIR, "best_params.json")
with open(best_params_path, encoding="utf-8") as f:
    best_params: dict[str, Any] = json.load(f)
print(f"\n[3] best_params 로드 완료 → {best_params_path}")
print(f"    params: {best_params}")

## 7. 시드별 OOF 학습

In [ ]:
# ── 시드별 OOF 학습 ───────────────────────────────────────
print(f"\n[4] 시드 앙상블 시작 | 시드 목록: {ENSEMBLE_SEEDS}")
print("=" * 65)

all_oof_probas: list[NDArray[np.float64]] = []
all_test_probas: list[NDArray[np.float64]] = []
seed_results = []

for seed_idx, seed in enumerate(ENSEMBLE_SEEDS, 1):
    print(f"\n  ▶ Seed {seed} ({seed_idx}/{len(ENSEMBLE_SEEDS)})")
    oof_proba, fold_models = train_single_seed(best_params, X_train, y_train, ratio, seed)
    all_oof_probas.append(oof_proba)

    # test proba: fold 모델 평균
    test_probas = np.column_stack([m.predict_proba(X_test)[:, 1] for m in fold_models])
    test_proba_seed: NDArray[np.float64] = test_probas.mean(axis=1)
    all_test_probas.append(test_proba_seed)

    # 단일 시드 OOF 성능 확인
    thr, f1, recall, prec = tune_threshold(oof_proba, y_train.values)
    seed_results.append(
        {
            "seed": seed,
            "oof_threshold": thr,
            "oof_recall": round(recall, 4),
            "oof_precision": round(prec, 4),
            "oof_f1": round(f1, 4),
            "oof_auc": round(float(roc_auc_score(y_train, oof_proba)), 4),
        }
    )
    print(f"    → OOF (thr={thr:.2f}) | AUC: {seed_results[-1]['oof_auc']:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

    # 시드별 모델 저장
    seed_dir = os.path.join(OUTPUT_DIR, f"seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)
    for i, m in enumerate(fold_models, 1):
        m.save_model(os.path.join(seed_dir, f"model_fold{i}.cbm"))
    np.save(os.path.join(seed_dir, "oof_proba.npy"), oof_proba)

## 8. 앙상블 OOF proba 계산 & Threshold 탐색

In [ ]:
# ── 앙상블 OOF proba (단순 평균) ─────────────────────────
print(f"\n[5] 앙상블 OOF proba 계산 (시드 {len(ENSEMBLE_SEEDS)}개 평균)")
ens_oof_proba: NDArray[np.float64] = np.mean(all_oof_probas, axis=0)
ens_test_proba: NDArray[np.float64] = np.mean(all_test_probas, axis=0)

# ── 앙상블 OOF Threshold 탐색 ────────────────────────────
best_thr, best_f1, oof_recall, oof_prec = tune_threshold(ens_oof_proba, y_train.values)
ens_oof_auc = float(roc_auc_score(y_train, ens_oof_proba))
print(f"    앙상블 OOF | AUC: {ens_oof_auc:.4f}")
print(f"    최적 threshold: {best_thr:.2f} | Recall: {oof_recall:.4f} | Precision: {oof_prec:.4f} | F1: {best_f1:.4f}")

## 9. 단일 시드 vs 앙상블 비교

In [ ]:
# ── 단일 시드 vs 앙상블 OOF 비교 ─────────────────────────
print(f"\n[6] 단일 시드 vs 앙상블 OOF 비교 (RECALL_MIN={RECALL_MIN})")
print("=" * 65)
print(f"  {'구분':<18} {'AUC':>7} {'Recall':>8} {'Precision':>10} {'F1':>8} {'THR':>6}")
print("-" * 65)
for r in seed_results:
    print(
        f"  Seed {r['seed']:<12} {r['oof_auc']:>7.4f} {r['oof_recall']:>8.4f} "
        f"{r['oof_precision']:>10.4f} {r['oof_f1']:>8.4f} {r['oof_threshold']:>6.2f}"
    )
print("-" * 65)
print(
    f"  {'앙상블 평균':<18} {ens_oof_auc:>7.4f} {oof_recall:>8.4f} {oof_prec:>10.4f} {best_f1:>8.4f} {best_thr:>6.2f}"
)
print("=" * 65)

## 10. 최종 Hold-out Test 평가

In [ ]:
# ── 최종 Hold-out Test 평가 ───────────────────────────────
print(f"\n[7] 최종 Hold-out Test 평가 (threshold={best_thr:.2f})")
print("=" * 65)
test_label = (ens_test_proba >= best_thr).astype(int)
test_auc = float(roc_auc_score(y_test, ens_test_proba))
test_recall = float(recall_score(y_test, test_label))
test_prec = float(precision_score(y_test, test_label, zero_division=0))
test_f1 = float(f1_score(y_test, test_label))
cm = confusion_matrix(y_test, test_label)

print(f"    AUC       : {test_auc:.4f}")
print(f"    Recall    : {test_recall:.4f}")
print(f"    Precision : {test_prec:.4f}")
print(f"    F1        : {test_f1:.4f}")
print(f"    TN={cm[0, 0]}  FP={cm[0, 1]}")
print(f"    FN={cm[1, 0]}  TP={cm[1, 1]}")
print("\n[7] Classification Report")
print(classification_report(y_test, test_label, target_names=["정상(0)", "이상지질혈증(1)"]))

## 11. 저장

In [ ]:
# ── 저장 ─────────────────────────────────────────────────
print(f"\n[8] 저장 중 → {OUTPUT_DIR}")

# 시드별 결과 CSV
seed_df = pd.DataFrame(seed_results)
seed_df.loc[len(seed_df)] = {
    "seed": "ensemble",
    "oof_threshold": best_thr,
    "oof_recall": round(oof_recall, 4),
    "oof_precision": round(oof_prec, 4),
    "oof_f1": round(best_f1, 4),
    "oof_auc": round(ens_oof_auc, 4),
}
seed_df.to_csv(os.path.join(OUTPUT_DIR, "seed_comparison.csv"), index=False)

# 앙상블 OOF proba 저장
np.save(os.path.join(OUTPUT_DIR, "ens_oof_proba.npy"), ens_oof_proba)
np.save(os.path.join(OUTPUT_DIR, "ens_oof_y_true.npy"), y_train.values)
np.save(os.path.join(OUTPUT_DIR, "ens_test_proba.npy"), ens_test_proba)
np.save(os.path.join(OUTPUT_DIR, "best_threshold.npy"), np.array([best_thr]))

# 최종 결과 JSON
import json

final_result = {
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "best_threshold": best_thr,
    "oof": {
        "auc": round(ens_oof_auc, 4),
        "recall": round(oof_recall, 4),
        "precision": round(oof_prec, 4),
        "f1": round(best_f1, 4),
    },
    "test": {
        "auc": round(test_auc, 4),
        "recall": round(test_recall, 4),
        "precision": round(test_prec, 4),
        "f1": round(test_f1, 4),
        "confusion_matrix": cm.tolist(),
    },
}
with open(os.path.join(OUTPUT_DIR, "ensemble_result.json"), "w", encoding="utf-8") as f:
    json.dump(final_result, f, ensure_ascii=False, indent=2)

print("    저장 완료:")
print("    - seed_comparison.csv   (시드별 OOF 성능 비교)")
print("    - ens_oof_proba.npy     (앙상블 OOF proba)")
print("    - ens_test_proba.npy    (앙상블 Test proba)")
print("    - best_threshold.npy    (최적 threshold)")
print("    - ensemble_result.json  (최종 결과)")
print("    - seed_*/               (시드별 fold 모델 + OOF proba)")
print(f"\n[완료] 시드 앙상블 ({len(ENSEMBLE_SEEDS)}개) 종료")